
# ARTI 308 – Lab 5: Feature Engineering (Classification)
## Order Status Prediction using a Talabat-style Orders Dataset

This notebook follows the same structure as the provided lab file but includes **improved feature engineering and experiments** using the actual dataset.


## 1. Setup and imports

In [ ]:

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif


## 2. Load the dataset

In [ ]:

DATA_PATH = "talabat_enhanced_orders.csv"
df = pd.read_csv(DATA_PATH)

df.head()


## 3. Basic dataset checks

In [ ]:

print("Shape:", df.shape)
print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())


Dataset is clean so we focus mainly on feature engineering.

## 4. Target variable distribution

In [ ]:

df["Order_Status"].value_counts(normalize=True)


## 5. Task 1 – Engineered Feature

In [ ]:

# Feature 1: item popularity
item_counts = df["Item_Name"].value_counts()
df["item_popularity"] = df["Item_Name"].map(item_counts)

# Feature 2: distance efficiency (price per km)
df["price_per_km"] = df["Total_Price"] / (df["Delivery_Distance_km"] + 0.01)

df[["Item_Name","item_popularity","price_per_km"]].head()



Two engineered features were created:

**item_popularity** – counts how frequently each item appears in the dataset.  
Popular items are likely prepared more efficiently and therefore more likely to be delivered successfully.

**price_per_km** – measures the relationship between order value and delivery distance.  
Expensive orders with very long distances may have a higher chance of cancellation or delay.


## 6. Task 2 – Alternative Peak Hour Rule

In [ ]:

df["order_hour"] = pd.to_datetime(df["Order_Time"]).dt.hour

def is_peak_hour(hour):
    return 1 if (11 <= hour <= 14) or (17 <= hour <= 21) else 0

df["is_peak_hour"] = df["order_hour"].apply(is_peak_hour)

df[["order_hour","is_peak_hour"]].head()



Peak hours were expanded to **11–14 and 17–21** instead of narrower windows to better capture the lunch and dinner rush.


## 7. Task 3 – Item_Name reduction experiments

In [ ]:

def reduce_items(data, k):
    top_items = data["Item_Name"].value_counts().nlargest(k).index
    data["Item_Name_reduced"] = data["Item_Name"].apply(lambda x: x if x in top_items else "Other")
    return data

results = {}

for k in [10, 30, 50]:
    
    temp = df.copy()
    temp = reduce_items(temp, k)
    
    features = [
        "Item_Name_reduced",
        "Payment_Method",
        "Traffic_Level",
        "Driver_Availability",
        "item_popularity",
        "price_per_km",
        "is_peak_hour",
        "Delivery_Distance_km"
    ]
    
    X = temp[features]
    y = temp["Order_Status"]
    
    cat = ["Item_Name_reduced","Payment_Method","Traffic_Level","Driver_Availability"]
    num = ["item_popularity","price_per_km","is_peak_hour","Delivery_Distance_km"]
    
    pre = ColumnTransformer([
        ("cat",OneHotEncoder(handle_unknown="ignore"),cat),
        ("num","passthrough",num)
    ])
    
    model = Pipeline([
        ("prep",pre),
        ("clf",RandomForestClassifier(n_estimators=120,random_state=42))
    ])
    
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
    
    model.fit(X_train,y_train)
    
    preds = model.predict(X_test)
    
    results[k] = accuracy_score(y_test,preds)

results



We compare model accuracy when reducing item categories to the **top 10, top 30, and top 50** most common items.


## 8. Task 4 – Feature Selection

In [ ]:

features = [
    "Payment_Method",
    "Traffic_Level",
    "Driver_Availability",
    "item_popularity",
    "price_per_km",
    "is_peak_hour",
    "Delivery_Distance_km"
]

X = df[features]
y = df["Order_Status"]

cat = ["Payment_Method","Traffic_Level","Driver_Availability"]
num = ["item_popularity","price_per_km","is_peak_hour","Delivery_Distance_km"]

pre = ColumnTransformer([
    ("cat",OneHotEncoder(handle_unknown="ignore"),cat),
    ("num","passthrough",num)
])

X_processed = pre.fit_transform(X)

selector = SelectKBest(score_func=f_classif, k=6)
X_selected = selector.fit_transform(X_processed, y)

model = RandomForestClassifier(random_state=42)
model.fit(X_selected,y)

preds = model.predict(X_selected)

accuracy_score(y,preds)



Feature selection slightly reduces dimensionality by keeping the most informative variables.
In our experiment, the performance difference is small, indicating most engineered features already contribute useful information.
